# Evaluation 03 — Six-class synthetic morphology QA

Upload a manually reviewed generation manifest. This is a documentation and gating notebook, **not** an automatic medical diagnostic tool. Test on all six classes and every generator mode.

In [ ]:
# Common Kaggle bootstrap: either attach DermSynReal source as Kaggle Input,
# or set CLONE_REPO_IF_MISSING=True and enable Kaggle Internet.
from pathlib import Path
import subprocess,sys,os,json
REPO_ROOT=Path("/kaggle/input/EDIT_REPO_DATASET/DermSynReal")
CLONE_REPO_IF_MISSING=True
EXPECTED_GIT_SHA=""  # Recommended: set a full pinned SHA for reproducible final runs.
if not (REPO_ROOT/"src/dermsynreal/evaluation.py").is_file():
    if not CLONE_REPO_IF_MISSING:
        raise FileNotFoundError("Attach the repository and change REPO_ROOT, or enable Git clone.")
    REPO_ROOT=Path("/kaggle/working/DermSynReal")
    if not (REPO_ROOT/".git").is_dir():
        subprocess.check_call(["git","clone","https://github.com/shanmukhdatta/DermSynReal.git",str(REPO_ROOT)])
if EXPECTED_GIT_SHA:
    if not (REPO_ROOT/".git").is_dir():
        raise ValueError("For pinned SHA verification attach a Git checkout or enable clone")
    actual=subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip()
    if actual!=EXPECTED_GIT_SHA:
        raise ValueError(f"Repository SHA {actual} differs from pinned SHA {EXPECTED_GIT_SHA}")
elif (REPO_ROOT/".git").is_dir():
    print("Current Git SHA:",subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip())
    print("WARNING: pin EXPECTED_GIT_SHA for final research outputs")
sys.path.insert(0,str(REPO_ROOT/"src"))
print("DermSynReal source:",REPO_ROOT)


In [ ]:
REVIEWS_CSV="/kaggle/input/EDIT_REVIEW_DATA/generation_reviews.csv"
OUTPUT_DIR="/kaggle/working/dermsynreal_eval_03"
RUN_QUALITY_AUDIT=False
# Required CSV columns:
# image_id,class_name,prompt_id,prompt_family_id,generator_id,
# reviewer_id,decision,morphology_score,near_copy_flag
# Decision: accepted/rejected/needs_review. Score 0..4. No accepted near-copy flags.

In [ ]:
from dermsynreal.quality_audit import summarize_quality
if RUN_QUALITY_AUDIT:
    summary=summarize_quality(REVIEWS_CSV,OUTPUT_DIR)
    print(json.dumps(summary,indent=2))
    import pandas as pd
    frame=pd.read_csv(Path(OUTPUT_DIR)/"generation_quality_counts.csv")
    display(frame)
    # Plots show only audited observations, never generated placeholder values.
    import matplotlib.pyplot as plt
    plot=frame.set_index(["class_name","generator_id"])[["accepted","rejected","needs_review"]]
    ax=plot.plot(kind="bar",stacked=True,figsize=(12,5))
    ax.set_ylabel("Reviewed images")
    ax.set_title("Generation QA by class and generator (not clinical accuracy)")
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR)/"generation_quality_counts.png",dpi=170)
    plt.show()
else:
    print("No QA results computed. Attach a reviewed CSV and set RUN_QUALITY_AUDIT=True.")


This does not establish clinician-grade authenticity, fairness, or training usefulness. Reject memorized or suspicious near-copies; do not treat high aesthetic realism as diagnosis correctness.